# Transformer Architecture: Pre-Training & Inference
## A Step-by-Step Matrix-Level Walkthrough (BERT with GPT Differences)

---

## Overview

This notebook traces a **single input** through every layer of a Transformer encoder (BERT-style), showing the **exact matrix shapes and operations** at each stage. Where GPT (a decoder-only model) differs, the change is noted as a **🔀 GPT Difference** sub-step.

---

## Running Example — Assumptions & Hyperparameters

| Symbol | Meaning | Value |
|--------|---------|-------|
| $$B$$ | Batch size | 1 |
| $$T$$ | Max sequence length (number of tokens) | 10 |
| $$d$$ | Embedding / hidden dimension | 784 |
| $$d_{ff}$$ | Feed-forward intermediate dimension | 3136 $$(= 4 \times d)$$ |
| $$V$$ | Vocabulary size | 30,522 |
| $$h$$ | Number of attention heads | 8 |
| $$d_k$$ | Per-head dimension $$(= d / h)$$ | 98 |
| $$N_{seg}$$ | Number of segment (token-type) IDs | 2 |

---

## Architecture Flowchart (with shapes)

```
Token IDs                          [B, T]         = [1, 10]
     │
     ▼
Token Embedding                    [B, T, d]      = [1, 10, 784]
     +
Position Embedding                 [1, T, d]      = [1, 10, 784]
     +
Token-Type Embedding               [B, T, d]      = [1, 10, 784]
     │
     ▼
Input Representation X             [B, T, d]      = [1, 10, 784]
     │
     ▼
Linear → Q, K, V                   [B, h, T, d_k] = [1, 8, 10, 98]
     │
     ▼
Scaled Dot-Product Attention       [B, h, T, T]   = [1, 8, 10, 10]
     │
     ▼
Concat + Project                   [B, T, d]      = [1, 10, 784]
     │
     ▼
Residual + LayerNorm               [B, T, d]      = [1, 10, 784]
     │
     ▼
FFN: 784 → 3136 → 784             [B, T, d]      = [1, 10, 784]
     │
     ▼
Residual + LayerNorm               [B, T, d]      = [1, 10, 784]
     │
     ▼
Contextual Representations         [B, T, d]      = [1, 10, 784]
     │
     ▼
MLM Head (Dense + GELU + LN)       [B, T, d]      = [1, 10, 784]
     │
     ▼
Vocabulary Projection              [B, T, V]      = [1, 10, 30522]
     │
     ▼
Softmax                            [B, T, V]      = [1, 10, 30522]
     │
     ▼
Predicted Masked Token(s)          argmax per position
```

---

## Key Conventions

* All weight matrices are **randomly initialized** (simulating an untrained model)
* We trace shapes explicitly at every step to build intuition
* BERT = Bidirectional Encoder Representations from Transformers (encoder-only)
* GPT = Generative Pre-trained Transformer (decoder-only)

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ============================================================
# HYPERPARAMETERS (fixed for the entire notebook)
# ============================================================
torch.manual_seed(42)
np.random.seed(42)

B = 1           # Batch size
T = 10          # Max sequence length
d = 784         # Embedding / hidden dimension
d_ff = 3136     # FFN intermediate dimension (4 * d)
V = 30522       # Vocabulary size (BERT WordPiece vocab)
h = 8           # Number of attention heads
d_k = d // h    # Per-head dimension = 784 // 8 = 98
N_seg = 2       # Number of segment types (sentence A vs B)

print(f"Configuration:")
print(f"  Batch size (B)        = {B}")
print(f"  Sequence length (T)   = {T}")
print(f"  Hidden dim (d)        = {d}")
print(f"  FFN dim (d_ff)        = {d_ff}")
print(f"  Vocab size (V)        = {V}")
print(f"  Attention heads (h)   = {h}")
print(f"  Per-head dim (d_k)    = {d_k}")
print(f"  Segment types         = {N_seg}")

Configuration:
  Batch size (B)        = 1
  Sequence length (T)   = 10
  Hidden dim (d)        = 784
  FFN dim (d_ff)        = 3136
  Vocab size (V)        = 30522
  Attention heads (h)   = 8
  Per-head dim (d_k)    = 98
  Segment types         = 2


## Step 1: Token IDs

### What happens here

Raw text is converted to a sequence of **integer IDs** via a tokenizer (e.g., WordPiece for BERT, BPE for GPT). Each ID maps to a word or sub-word in a fixed vocabulary of size $$V = 30{,}522$$.

### Example sentence

```
Raw text:  "The cat [MASK] on the mat [SEP] It slept [SEP]"
```

After tokenization (assuming pre-tokenized for clarity):

| Position | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|----------|---|---|---|---|---|---|---|---|---|---|
| Token | [CLS] | The | cat | [MASK] | on | the | mat | [SEP] | It | [SEP] |
| ID | 101 | 1996 | 4937 | 103 | 2006 | 1996 | 13523 | 102 | 2009 | 102 |

### Matrix shape

$$\text{token\_ids} \in \mathbb{Z}^{B \times T} = \mathbb{Z}^{1 \times 10}$$

Each entry is an integer in $$[0, V-1]$$.

### 🔀 GPT Difference
GPT has **no `[CLS]` or `[SEP]`** tokens. It uses `<|endoftext|>` as a boundary token. The sequence is a single continuous stream (no sentence pairs).

In [0]:
# ==============================================================
# STEP 1: Token IDs
# ==============================================================
# Simulating tokenized input:
# [CLS] The cat [MASK] on the mat [SEP] It [SEP]
token_ids = torch.tensor([[101, 1996, 4937, 103, 2006, 1996, 13523, 102, 2009, 102]])

print(f"token_ids shape: {token_ids.shape}")   # [B, T] = [1, 10]
print(f"token_ids:\n{token_ids}")
print(f"\nEach integer is an index into a vocabulary of size V={V}")
print(f"Min ID: {token_ids.min().item()}, Max ID: {token_ids.max().item()}")

token_ids shape: torch.Size([1, 10])
token_ids:
tensor([[  101,  1996,  4937,   103,  2006,  1996, 13523,   102,  2009,   102]])

Each integer is an index into a vocabulary of size V=30522
Min ID: 101, Max ID: 13523


## Step 2: Token Embedding

### What happens here

A **lookup table** (embedding matrix) of shape $$V \times d = 30{,}522 \times 784$$ maps each token ID to a dense vector of dimension $$d = 784$$.

### Mathematical operation

Let $$\mathbf{E}_{\text{tok}} \in \mathbb{R}^{V \times d}$$ be the token embedding matrix (learnable parameters).

For each token ID $$x_i$$:

$$\mathbf{e}_i = \mathbf{E}_{\text{tok}}[x_i, :] \in \mathbb{R}^{d}$$

This is **not** a matrix multiplication — it is an **index lookup** (equivalent to multiplying a one-hot vector by the embedding matrix, but implemented efficiently via direct indexing).

### Shapes

$$\text{Input: token\_ids} \in \mathbb{Z}^{B \times T} = \mathbb{Z}^{1 \times 10}$$

$$\text{Embedding table: } \mathbf{E}_{\text{tok}} \in \mathbb{R}^{V \times d} = \mathbb{R}^{30522 \times 784}$$

$$\text{Output: token\_embeds} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Number of parameters

$$V \times d = 30{,}522 \times 784 = 23{,}929{,}248 \approx 24\text{M parameters}$$

This is often the **single largest parameter block** in the model!

In [0]:
# ==============================================================
# STEP 2: Token Embedding
# ==============================================================
# Create embedding table: V x d = 30522 x 784
token_embedding_table = nn.Embedding(num_embeddings=V, embedding_dim=d)

print(f"Embedding table shape: [{V}, {d}] = {list(token_embedding_table.weight.shape)}")
print(f"Number of parameters: {V * d:,}")

# Lookup: token_ids [1, 10] -> token_embeds [1, 10, 784]
token_embeds = token_embedding_table(token_ids)

print(f"\nInput shape:  {token_ids.shape}    (token IDs)")
print(f"Output shape: {token_embeds.shape}  (dense vectors)")
print(f"\nFirst token ([CLS], ID=101) embedding (first 10 dims):")
print(f"  {token_embeds[0, 0, :10].detach().numpy().round(4)}")

Embedding table shape: [30522, 784] = [30522, 784]
Number of parameters: 23,929,248

Input shape:  torch.Size([1, 10])    (token IDs)
Output shape: torch.Size([1, 10, 784])  (dense vectors)

First token ([CLS], ID=101) embedding (first 10 dims):
  [ 0.4261  1.2872 -0.9602  0.716   1.1435 -0.3726  0.2758  0.7135 -1.4819
  1.3054]


## Step 3: Position Embedding

### What happens here

Transformers have **no inherent notion of order** (unlike RNNs). Position embeddings inject sequential information by assigning a unique learnable vector to each position $$0, 1, \ldots, T-1$$.

### Mathematical operation

Let $$\mathbf{E}_{\text{pos}} \in \mathbb{R}^{T_{\max} \times d}$$ be the position embedding matrix.

For position $$p$$:

$$\mathbf{p}_p = \mathbf{E}_{\text{pos}}[p, :] \in \mathbb{R}^{d}$$

The position IDs are simply $$[0, 1, 2, \ldots, T-1]$$.

### Shapes

$$\text{Position IDs: } [0, 1, ..., 9] \in \mathbb{Z}^{T} = \mathbb{Z}^{10}$$

$$\text{Position embedding table: } \mathbf{E}_{\text{pos}} \in \mathbb{R}^{T \times d} = \mathbb{R}^{10 \times 784}$$

$$\text{Output: pos\_embeds} \in \mathbb{R}^{1 \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Important notes

* BERT uses **learned** position embeddings (a simple lookup table, just like token embeddings)
* The original Transformer paper (Vaswani et al., 2017) used **sinusoidal** (fixed) positional encodings
* GPT-2 also uses learned positional embeddings; GPT-3+ variants may use RoPE (Rotary Position Embedding)

### 🔀 GPT Difference
GPT uses the same learned positional embedding mechanism. No difference in this step (though the maximum sequence length $$T_{\max}$$ may differ).

In [0]:
# ==============================================================
# STEP 3: Position Embedding
# ==============================================================
# Position IDs: [0, 1, 2, ..., T-1]
position_ids = torch.arange(T).unsqueeze(0)  # Shape: [1, T] = [1, 10]

# Position embedding table: T x d = 10 x 784
position_embedding_table = nn.Embedding(num_embeddings=T, embedding_dim=d)

print(f"Position IDs: {position_ids}")
print(f"Position embedding table shape: [{T}, {d}] = {list(position_embedding_table.weight.shape)}")

# Lookup: position_ids [1, 10] -> pos_embeds [1, 10, 784]
pos_embeds = position_embedding_table(position_ids)

print(f"\nInput shape:  {position_ids.shape}    (position indices)")
print(f"Output shape: {pos_embeds.shape}  (positional vectors)")
print(f"\nPosition 0 embedding (first 10 dims):")
print(f"  {pos_embeds[0, 0, :10].detach().numpy().round(4)}")
print(f"Position 9 embedding (first 10 dims):")
print(f"  {pos_embeds[0, 9, :10].detach().numpy().round(4)}")
print(f"\n(Different vectors => model can distinguish positions)")

Position IDs: tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
Position embedding table shape: [10, 784] = [10, 784]

Input shape:  torch.Size([1, 10])    (position indices)
Output shape: torch.Size([1, 10, 784])  (positional vectors)

Position 0 embedding (first 10 dims):
  [ 0.3837  0.2189  0.7393  1.1904 -0.3363  1.2632 -0.0437 -1.2568 -0.4491
 -0.8828]
Position 9 embedding (first 10 dims):
  [ 0.0981  1.1207 -1.7788  0.1338 -1.2402 -0.4832 -1.6775  0.4625  0.494
 -0.2391]

(Different vectors => model can distinguish positions)


## Step 4: Token-Type (Segment) Embedding

### What happens here

BERT is pre-trained on **sentence pairs** (for the Next Sentence Prediction task). A **segment embedding** tells the model which sentence each token belongs to:

* Segment 0 = Sentence A (including `[CLS]` and the first `[SEP]`)
* Segment 1 = Sentence B (including the second `[SEP]`)

### Our example

| Position | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|----------|---|---|---|---|---|---|---|---|---|---|
| Token | [CLS] | The | cat | [MASK] | on | the | mat | [SEP] | It | [SEP] |
| Segment | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 |

### Mathematical operation

Let $$\mathbf{E}_{\text{seg}} \in \mathbb{R}^{N_{seg} \times d} = \mathbb{R}^{2 \times 784}$$ be the segment embedding matrix.

$$\text{seg\_embeds} = \mathbf{E}_{\text{seg}}[\text{segment\_ids}]$$

### Shapes

$$\text{Segment IDs} \in \{0, 1\}^{B \times T} = \{0, 1\}^{1 \times 10}$$

$$\text{Segment embedding table: } \mathbf{E}_{\text{seg}} \in \mathbb{R}^{2 \times 784}$$

$$\text{Output: seg\_embeds} \in \mathbb{R}^{1 \times 10 \times 784}$$

### 🔀 GPT Difference
GPT has **NO token-type/segment embedding**. Since GPT processes a single continuous sequence (no sentence pairs), this entire component is **skipped**. The input embedding is just:
$$\mathbf{X} = \mathbf{E}_{\text{tok}} + \mathbf{E}_{\text{pos}}$$ (GPT)
vs.
$$\mathbf{X} = \mathbf{E}_{\text{tok}} + \mathbf{E}_{\text{pos}} + \mathbf{E}_{\text{seg}}$$ (BERT)

In [0]:
# ==============================================================
# STEP 4: Token-Type (Segment) Embedding
# ==============================================================
# Sentence A: positions 0-7, Sentence B: positions 8-9
segment_ids = torch.tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1]])

# Segment embedding table: 2 x d = 2 x 784
segment_embedding_table = nn.Embedding(num_embeddings=N_seg, embedding_dim=d)

print(f"Segment IDs: {segment_ids}")
print(f"Segment embedding table shape: [{N_seg}, {d}] = {list(segment_embedding_table.weight.shape)}")
print(f"Number of parameters: {N_seg * d:,} (tiny!)")

# Lookup: segment_ids [1, 10] -> seg_embeds [1, 10, 784]
seg_embeds = segment_embedding_table(segment_ids)

print(f"\nInput shape:  {segment_ids.shape}")
print(f"Output shape: {seg_embeds.shape}")
print(f"\nSegment 0 embedding (first 10 dims):")
print(f"  {seg_embeds[0, 0, :10].detach().numpy().round(4)}")
print(f"Segment 1 embedding (first 10 dims):")
print(f"  {seg_embeds[0, 8, :10].detach().numpy().round(4)}")

Segment IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1]])
Segment embedding table shape: [2, 784] = [2, 784]
Number of parameters: 1,568 (tiny!)

Input shape:  torch.Size([1, 10])
Output shape: torch.Size([1, 10, 784])

Segment 0 embedding (first 10 dims):
  [ 0.0394  0.1986  0.0525  0.3736 -1.198  -0.9054  0.3207 -0.5133  0.2397
 -0.6132]
Segment 1 embedding (first 10 dims):
  [ 0.4522  1.4634 -2.0146 -1.664  -0.2248  0.0574  0.6873 -0.2877  0.84
  2.1107]


## Step 5: Combined Input Embedding

### What happens here

The three embeddings are **summed element-wise** to produce the final input representation. After summation, **LayerNorm** and **Dropout** are applied.

### Mathematical operation

$$\mathbf{X}_{\text{raw}} = \mathbf{E}_{\text{tok}}(\text{token\_ids}) + \mathbf{E}_{\text{pos}}(\text{position\_ids}) + \mathbf{E}_{\text{seg}}(\text{segment\_ids})$$

$$\mathbf{X} = \text{LayerNorm}(\mathbf{X}_{\text{raw}}) + \text{Dropout}$$

where LayerNorm normalizes each vector (across the $$d$$ dimension) to have zero mean and unit variance:

$$\text{LayerNorm}(\mathbf{x}) = \gamma \cdot \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

* $$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i$$ (mean across features)
* $$\sigma^2 = \frac{1}{d}\sum_{i=1}^{d}(x_i - \mu)^2$$ (variance across features)
* $$\gamma, \beta \in \mathbb{R}^d$$ are learnable scale and shift parameters
* $$\epsilon = 10^{-12}$$ (for numerical stability)

### Shapes

All three embeddings are $$\mathbb{R}^{1 \times 10 \times 784}$$, so addition is straightforward:

$$\text{Input to encoder: } \mathbf{X} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Why addition (not concatenation)?

Addition keeps the dimensionality at $$d$$. Concatenation would triple it to $$3d = 2352$$, making all downstream weight matrices larger. The information is "mixed" — each dimension carries a blend of token identity, position, and segment.

In [0]:
# ==============================================================
# STEP 5: Combined Input Embedding
# ==============================================================
# Element-wise sum of all three embeddings
X_raw = token_embeds + pos_embeds + seg_embeds
print(f"token_embeds shape: {token_embeds.shape}")
print(f"pos_embeds shape:   {pos_embeds.shape}")
print(f"seg_embeds shape:   {seg_embeds.shape}")
print(f"Sum (X_raw) shape:  {X_raw.shape}")

# Apply LayerNorm
layer_norm_emb = nn.LayerNorm(d, eps=1e-12)
X = layer_norm_emb(X_raw)

print(f"\nAfter LayerNorm:")
print(f"  X shape: {X.shape}")
print(f"  X[0,0,:5] before LN: {X_raw[0, 0, :5].detach().numpy().round(4)}")
print(f"  X[0,0,:5] after  LN: {X[0, 0, :5].detach().numpy().round(4)}")

# Verify LayerNorm properties (mean ≈ 0, std ≈ 1 per token)
with torch.no_grad():
    sample_vec = X[0, 0, :]  # First token's embedding
    print(f"\n  Mean of normalized vector: {sample_vec.mean().item():.6f} (should be ≈ 0)")
    print(f"  Std  of normalized vector: {sample_vec.std().item():.6f}  (should be ≈ 1)")

print(f"\n✓ Final input to Transformer encoder: X ∈ R^[{B}, {T}, {d}]")

token_embeds shape: torch.Size([1, 10, 784])
pos_embeds shape:   torch.Size([1, 10, 784])
seg_embeds shape:   torch.Size([1, 10, 784])
Sum (X_raw) shape:  torch.Size([1, 10, 784])

After LayerNorm:
  X shape: torch.Size([1, 10, 784])
  X[0,0,:5] before LN: [ 0.8492  1.7048 -0.1684  2.2799 -0.3907]
  X[0,0,:5] after  LN: [ 0.5045  0.9863 -0.0684  1.3101 -0.1936]

  Mean of normalized vector: 0.000000 (should be ≈ 0)
  Std  of normalized vector: 1.000638  (should be ≈ 1)

✓ Final input to Transformer encoder: X ∈ R^[1, 10, 784]


## Step 6: Q, K, V Linear Projections

### What happens here

Each input vector is linearly projected into three separate spaces: **Query (Q)**, **Key (K)**, and **Value (V)**. These projections enable the self-attention mechanism to compute "what to attend to" (Q·K) and "what information to extract" (V).

### Mathematical operation

For each token vector $$\mathbf{x}_i \in \mathbb{R}^d$$:

$$\mathbf{Q} = \mathbf{X} \cdot \mathbf{W}^Q + \mathbf{b}^Q$$

$$\mathbf{K} = \mathbf{X} \cdot \mathbf{W}^K + \mathbf{b}^K$$

$$\mathbf{V} = \mathbf{X} \cdot \mathbf{W}^V + \mathbf{b}^V$$

where:
* $$\mathbf{W}^Q, \mathbf{W}^K, \mathbf{W}^V \in \mathbb{R}^{d \times d} = \mathbb{R}^{784 \times 784}$$
* $$\mathbf{b}^Q, \mathbf{b}^K, \mathbf{b}^V \in \mathbb{R}^{d} = \mathbb{R}^{784}$$

### Multi-Head Reshaping

After projection, the $$d$$-dimensional vectors are **split** into $$h = 8$$ heads, each with dimension $$d_k = 98$$:

$$\mathbf{Q}_{\text{multi}} = \text{reshape}(\mathbf{Q}, [B, T, h, d_k]).\text{transpose}(1, 2) \in \mathbb{R}^{B \times h \times T \times d_k}$$

### Shape transformations

```
X:          [1, 10, 784]
              × W^Q [784, 784]
              ─────────────────
Q (flat):   [1, 10, 784]
  reshape → [1, 10, 8, 98]
  transpose→ [1, 8, 10, 98]    ← each head attends independently
```

### Intuition

* **Query**: "What am I looking for?"
* **Key**: "What do I contain that others might need?"
* **Value**: "What information do I actually provide?"

Each head learns a **different** type of relationship (e.g., one head might learn syntax, another semantics).

In [0]:
# ==============================================================
# STEP 6: Q, K, V Linear Projections
# ==============================================================
# Weight matrices for Q, K, V (each: d -> d)
W_Q = nn.Linear(d, d)  # W^Q: [784, 784] + bias [784]
W_K = nn.Linear(d, d)  # W^K: [784, 784] + bias [784]
W_V = nn.Linear(d, d)  # W^V: [784, 784] + bias [784]

print(f"W_Q weight shape: {list(W_Q.weight.shape)}, bias: {list(W_Q.bias.shape)}")
print(f"W_K weight shape: {list(W_K.weight.shape)}, bias: {list(W_K.bias.shape)}")
print(f"W_V weight shape: {list(W_V.weight.shape)}, bias: {list(W_V.bias.shape)}")
print(f"Parameters per projection: {d*d + d:,} = {d}×{d} + {d}")
print(f"Total Q,K,V parameters: {3*(d*d + d):,}")

# Project: X [1, 10, 784] × W [784, 784] -> Q,K,V [1, 10, 784]
Q_proj = W_Q(X)  # [B, T, d] = [1, 10, 784]
K_proj = W_K(X)  # [B, T, d] = [1, 10, 784]
V_proj = W_V(X)  # [B, T, d] = [1, 10, 784]

print(f"\nAfter linear projection:")
print(f"  Q shape: {Q_proj.shape}")
print(f"  K shape: {K_proj.shape}")
print(f"  V shape: {V_proj.shape}")

# Reshape for multi-head: [B, T, d] -> [B, T, h, d_k] -> [B, h, T, d_k]
Q_multi = Q_proj.view(B, T, h, d_k).transpose(1, 2)  # [1, 8, 10, 98]
K_multi = K_proj.view(B, T, h, d_k).transpose(1, 2)  # [1, 8, 10, 98]
V_multi = V_proj.view(B, T, h, d_k).transpose(1, 2)  # [1, 8, 10, 98]

print(f"\nAfter multi-head reshape:")
print(f"  Q_multi shape: {Q_multi.shape}  → [B, h, T, d_k]")
print(f"  K_multi shape: {K_multi.shape}  → [B, h, T, d_k]")
print(f"  V_multi shape: {V_multi.shape}  → [B, h, T, d_k]")
print(f"\n  Each of {h} heads operates on {d_k}-dim slices independently")

W_Q weight shape: [784, 784], bias: [784]
W_K weight shape: [784, 784], bias: [784]
W_V weight shape: [784, 784], bias: [784]
Parameters per projection: 615,440 = 784×784 + 784
Total Q,K,V parameters: 1,846,320

After linear projection:
  Q shape: torch.Size([1, 10, 784])
  K shape: torch.Size([1, 10, 784])
  V shape: torch.Size([1, 10, 784])

After multi-head reshape:
  Q_multi shape: torch.Size([1, 8, 10, 98])  → [B, h, T, d_k]
  K_multi shape: torch.Size([1, 8, 10, 98])  → [B, h, T, d_k]
  V_multi shape: torch.Size([1, 8, 10, 98])  → [B, h, T, d_k]

  Each of 8 heads operates on 98-dim slices independently


## Step 7: Scaled Dot-Product Self-Attention

### What happens here

This is the **core mechanism** of the Transformer. Each token computes attention scores with **every other token**, determining how much to "attend" to each position.

### Mathematical operation

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) \cdot V$$

**Step-by-step breakdown:**

**1. Compute raw attention scores:**
$$\text{scores} = \frac{Q \cdot K^\top}{\sqrt{d_k}} \in \mathbb{R}^{B \times h \times T \times T}$$

$$[1, 8, 10, 98] \times [1, 8, 98, 10] = [1, 8, 10, 10]$$

Division by $$\sqrt{d_k} = \sqrt{98} \approx 9.9$$ prevents the dot products from growing too large (which would push softmax into saturation).

**2. Apply attention mask (if needed):**
* **BERT**: No mask needed — every token attends to every other token (bidirectional)
* **GPT**: Apply a **causal mask** — token at position $$i$$ can only attend to positions $$\leq i$$

**3. Softmax normalization:**
$$\text{attn\_weights} = \text{softmax}(\text{scores}, \text{dim}=-1) \in \mathbb{R}^{B \times h \times T \times T}$$

Each row sums to 1 — it's a probability distribution over positions.

**4. Weighted sum of values:**
$$\text{attn\_output} = \text{attn\_weights} \cdot V \in \mathbb{R}^{B \times h \times T \times d_k}$$

$$[1, 8, 10, 10] \times [1, 8, 10, 98] = [1, 8, 10, 98]$$

**5. Concatenate heads and project:**
$$\text{concat} = \text{reshape}(\text{attn\_output}, [B, T, d]) \in \mathbb{R}^{1 \times 10 \times 784}$$
$$\text{output} = \text{concat} \cdot W^O + b^O$$

where $$W^O \in \mathbb{R}^{d \times d} = \mathbb{R}^{784 \times 784}$$

### 🔀 GPT Difference — Causal (Masked) Attention

GPT uses a **lower-triangular mask** to enforce autoregressive generation:

$$\text{mask}[i][j] = \begin{cases} 0 & \text{if } j \leq i \text{ (attend)} \\ -\infty & \text{if } j > i \text{ (block)} \end{cases}$$

This means position 3 can see positions 0, 1, 2, 3 but **not** 4, 5, ..., 9.

After softmax, $$-\infty$$ entries become 0, so future tokens contribute nothing.

This is the **only architectural difference** between BERT and GPT in the encoder/decoder block.

In [0]:
# ==============================================================
# STEP 7: Scaled Dot-Product Self-Attention
# ==============================================================
import math

# --- Step 7a: Compute attention scores ---
# Q_multi: [1, 8, 10, 98], K_multi^T: [1, 8, 98, 10]
scores = torch.matmul(Q_multi, K_multi.transpose(-2, -1))  # [1, 8, 10, 10]
print(f"Raw scores shape: {scores.shape}  (Q @ K^T)")

# Scale by sqrt(d_k)
scale_factor = math.sqrt(d_k)
scores_scaled = scores / scale_factor
print(f"Scale factor: sqrt({d_k}) = {scale_factor:.4f}")
print(f"Scaled scores shape: {scores_scaled.shape}")

# --- Step 7b: BERT - No mask (bidirectional) ---
print(f"\n--- BERT: Bidirectional (no mask) ---")
attn_weights_bert = F.softmax(scores_scaled, dim=-1)  # [1, 8, 10, 10]
print(f"Attention weights shape: {attn_weights_bert.shape}")
print(f"Row 0 sums to: {attn_weights_bert[0, 0, 0, :].sum().item():.4f} (should be 1.0)")
print(f"Sample attn weights (head 0, token 0 attending to all):")
print(f"  {attn_weights_bert[0, 0, 0, :].detach().numpy().round(4)}")

# --- Step 7b (GPT): Causal mask ---
print(f"\n--- GPT: Causal mask (lower triangular) ---")
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # Upper triangle = True
print(f"Causal mask (True = blocked):")
print(causal_mask.int().numpy())

scores_gpt = scores_scaled.clone()
scores_gpt = scores_gpt.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
attn_weights_gpt = F.softmax(scores_gpt, dim=-1)
print(f"\nGPT attn weights (head 0, token 3 attending to all):")
print(f"  {attn_weights_gpt[0, 0, 3, :].detach().numpy().round(4)}")
print(f"  (positions 4-9 are 0 → cannot see future!)")

# --- Step 7c: Weighted sum of values (using BERT weights) ---
attn_output = torch.matmul(attn_weights_bert, V_multi)  # [1, 8, 10, 98]
print(f"\n--- Weighted value output ---")
print(f"attn_output shape: {attn_output.shape}  [B, h, T, d_k]")

# --- Step 7d: Concatenate heads ---
# [1, 8, 10, 98] -> transpose -> [1, 10, 8, 98] -> reshape -> [1, 10, 784]
attn_concat = attn_output.transpose(1, 2).contiguous().view(B, T, d)
print(f"After concat heads: {attn_concat.shape}  [B, T, h*d_k = d]")

# --- Step 7e: Output projection ---
W_O = nn.Linear(d, d)  # [784, 784]
attn_final = W_O(attn_concat)  # [1, 10, 784]
print(f"After output projection (W^O): {attn_final.shape}")
print(f"\n✓ Self-attention complete: {list(attn_final.shape)}")

Raw scores shape: torch.Size([1, 8, 10, 10])  (Q @ K^T)
Scale factor: sqrt(98) = 9.8995
Scaled scores shape: torch.Size([1, 8, 10, 10])

--- BERT: Bidirectional (no mask) ---
Attention weights shape: torch.Size([1, 8, 10, 10])
Row 0 sums to: 1.0000 (should be 1.0)
Sample attn weights (head 0, token 0 attending to all):
  [0.096  0.0854 0.1306 0.1016 0.1246 0.0762 0.1114 0.1204 0.0666 0.0873]

--- GPT: Causal mask (lower triangular) ---
Causal mask (True = blocked):
[[0 1 1 1 1 1 1 1 1 1]
 [0 0 1 1 1 1 1 1 1 1]
 [0 0 0 1 1 1 1 1 1 1]
 [0 0 0 0 1 1 1 1 1 1]
 [0 0 0 0 0 1 1 1 1 1]
 [0 0 0 0 0 0 1 1 1 1]
 [0 0 0 0 0 0 0 1 1 1]
 [0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0]]

GPT attn weights (head 0, token 3 attending to all):
  [0.2411 0.2854 0.2435 0.2301 0.     0.     0.     0.     0.     0.    ]
  (positions 4-9 are 0 → cannot see future!)

--- Weighted value output ---
attn_output shape: torch.Size([1, 8, 10, 98])  [B, h, T, d_k]
After concat heads: torch.Size([1

## Step 8: Residual Connection + LayerNorm (Post-Attention)

### What happens here

The attention output is combined with the **original input** via a residual (skip) connection, then normalized with LayerNorm. This is the "Add & Norm" block.

### Mathematical operation

$$\mathbf{H}_{\text{attn}} = \text{LayerNorm}(\mathbf{X} + \text{Attention}(\mathbf{X}))$$

Expanded:

$$\mathbf{H}_{\text{attn}} = \text{LayerNorm}(\underbrace{\mathbf{X}}_{\text{residual}} + \underbrace{\text{MultiHeadAttn}(\mathbf{X})}_{\text{attention output}})$$

### Why residual connections?

1. **Gradient flow**: Gradients flow directly through the skip connection, preventing vanishing gradients in deep networks
2. **Identity initialization**: At initialization (random weights), the attention output is essentially noise. The residual ensures the input passes through relatively unmodified early in training
3. **Enables depth**: Without residuals, stacking 12+ layers would be impractical

### Shape (unchanged)

$$\mathbf{H}_{\text{attn}} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Note on Pre-LN vs Post-LN

* **BERT (Post-LN)**: $$\text{LN}(x + \text{sublayer}(x))$$ — normalize after addition
* **GPT-2+ (Pre-LN)**: $$x + \text{sublayer}(\text{LN}(x))$$ — normalize before sublayer

Pre-LN is more stable for training deep models. We show Post-LN (original Transformer/BERT style) here.

In [0]:
# ==============================================================
# STEP 8: Residual Connection + LayerNorm (Post-Attention)
# ==============================================================
layer_norm_1 = nn.LayerNorm(d, eps=1e-12)

# Residual connection: add input X back to attention output
residual_1 = X + attn_final  # [1, 10, 784] + [1, 10, 784]
print(f"Input X shape:           {X.shape}")
print(f"Attention output shape:  {attn_final.shape}")
print(f"After residual addition: {residual_1.shape}")

# LayerNorm
H_attn = layer_norm_1(residual_1)  # [1, 10, 784]
print(f"After LayerNorm:         {H_attn.shape}")

# Verify normalization
with torch.no_grad():
    sample = H_attn[0, 0, :]  # First token
    print(f"\n  Post-LN mean: {sample.mean().item():.6f} (≈ 0)")
    print(f"  Post-LN std:  {sample.std().item():.6f}  (≈ 1)")

print(f"\n✓ Post-attention sublayer complete: H_attn ∈ R^{list(H_attn.shape)}")

Input X shape:           torch.Size([1, 10, 784])
Attention output shape:  torch.Size([1, 10, 784])
After residual addition: torch.Size([1, 10, 784])
After LayerNorm:         torch.Size([1, 10, 784])

  Post-LN mean: 0.000000 (≈ 0)
  Post-LN std:  1.000638  (≈ 1)

✓ Post-attention sublayer complete: H_attn ∈ R^[1, 10, 784]


## Step 9: Position-wise Feed-Forward Network (FFN)

### What happens here

Each token's representation is passed through a **two-layer MLP** independently (same weights, applied position-by-position). This is where the model does its "thinking" — the attention layer routes information, and the FFN processes it.

### Mathematical operation

$$\text{FFN}(\mathbf{x}) = \text{GELU}(\mathbf{x} \cdot W_1 + b_1) \cdot W_2 + b_2$$

where:
* $$W_1 \in \mathbb{R}^{d \times d_{ff}} = \mathbb{R}^{784 \times 3136}$$ (expansion)
* $$b_1 \in \mathbb{R}^{d_{ff}} = \mathbb{R}^{3136}$$
* $$W_2 \in \mathbb{R}^{d_{ff} \times d} = \mathbb{R}^{3136 \times 784}$$ (compression)
* $$b_2 \in \mathbb{R}^{d} = \mathbb{R}^{784}$$

### Shape transformations

```
H_attn:        [1, 10, 784]
    × W_1 [784, 3136]
    ──────────────────
Intermediate:  [1, 10, 3136]    ← 4× expansion ("bottleneck" in reverse)
    GELU activation
    × W_2 [3136, 784]
    ──────────────────
FFN output:    [1, 10, 784]     ← back to original dimension
```

### GELU Activation

GELU (Gaussian Error Linear Unit) is a smooth approximation of ReLU:

$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

Unlike ReLU (hard cutoff at 0), GELU allows small negative values through, making optimization smoother.

### Parameters in the FFN

$$\text{FFN params} = (d \times d_{ff} + d_{ff}) + (d_{ff} \times d + d) = 784 \times 3136 + 3136 + 3136 \times 784 + 784$$
$$= 2{,}458{,}624 + 3{,}136 + 2{,}458{,}624 + 784 = 4{,}921{,}168$$

The FFN has **more parameters** than the attention layer! (Attention: $$4 \times d^2 \approx 2.5\text{M}$$ vs FFN: $$\approx 4.9\text{M}$$)

### Why expand then compress?

The 4× expansion allows the model to compute in a **higher-dimensional space** where linear separability is easier (per the kernel trick intuition), then project back to $$d$$ for the next layer.

In [0]:
# ==============================================================
# STEP 9: Feed-Forward Network (784 → 3136 → 784)
# ==============================================================
# Two linear layers with GELU activation in between
FFN_W1 = nn.Linear(d, d_ff)     # [784, 3136] + bias [3136]
FFN_W2 = nn.Linear(d_ff, d)     # [3136, 784] + bias [784]

print(f"FFN Layer 1: {list(FFN_W1.weight.shape)} + bias {list(FFN_W1.bias.shape)}")
print(f"FFN Layer 2: {list(FFN_W2.weight.shape)} + bias {list(FFN_W2.bias.shape)}")
print(f"Total FFN parameters: {(d*d_ff + d_ff) + (d_ff*d + d):,}")

# Forward pass through FFN
# Step 1: Expand 784 -> 3136
intermediate = FFN_W1(H_attn)  # [1, 10, 3136]
print(f"\nAfter W1 (expansion):  {intermediate.shape}")

# Step 2: GELU activation
intermediate_activated = F.gelu(intermediate)  # [1, 10, 3136]
print(f"After GELU activation: {intermediate_activated.shape}")

# Step 3: Compress 3136 -> 784
ffn_output = FFN_W2(intermediate_activated)  # [1, 10, 784]
print(f"After W2 (compression): {ffn_output.shape}")

# Demonstrate GELU vs ReLU on a sample
print(f"\n--- GELU vs ReLU comparison ---")
sample_vals = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])
print(f"Input:  {sample_vals.numpy()}")
print(f"ReLU:   {F.relu(sample_vals).numpy()}")
print(f"GELU:   {F.gelu(sample_vals).numpy().round(4)}")
print(f"(GELU allows small negative gradients to flow through)")

print(f"\n✓ FFN complete: {list(ffn_output.shape)}")

FFN Layer 1: [3136, 784] + bias [3136]
FFN Layer 2: [784, 3136] + bias [784]
Total FFN parameters: 4,921,168

After W1 (expansion):  torch.Size([1, 10, 3136])
After GELU activation: torch.Size([1, 10, 3136])
After W2 (compression): torch.Size([1, 10, 784])

--- GELU vs ReLU comparison ---
Input:  [-2.  -1.  -0.5  0.   0.5  1.   2. ]
ReLU:   [0.  0.  0.  0.  0.5 1.  2. ]
GELU:   [-0.0455 -0.1587 -0.1543  0.      0.3457  0.8413  1.9545]
(GELU allows small negative gradients to flow through)

✓ FFN complete: [1, 10, 784]


## Step 10: Residual Connection + LayerNorm (Post-FFN)

### What happens here

Identical pattern to Step 8 — the FFN output is added to its **input** (the post-attention representation) via a skip connection, then normalized.

### Mathematical operation

$$\mathbf{H}_{\text{out}} = \text{LayerNorm}(\mathbf{H}_{\text{attn}} + \text{FFN}(\mathbf{H}_{\text{attn}}))$$

### Shape (unchanged)

$$\mathbf{H}_{\text{out}} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Complete Transformer Block Summary

One Transformer encoder block thus computes:

$$\mathbf{H}' = \text{LN}(\mathbf{X} + \text{MHA}(\mathbf{X}))$$
$$\mathbf{H}_{\text{out}} = \text{LN}(\mathbf{H}' + \text{FFN}(\mathbf{H}'))$$

BERT-base stacks $$N = 12$$ such blocks. Each block's output feeds as input to the next.

In [0]:
# ==============================================================
# STEP 10: Residual Connection + LayerNorm (Post-FFN)
# ==============================================================
layer_norm_2 = nn.LayerNorm(d, eps=1e-12)

# Residual: add FFN input (H_attn) back to FFN output
residual_2 = H_attn + ffn_output  # [1, 10, 784] + [1, 10, 784]
print(f"H_attn (FFN input) shape: {H_attn.shape}")
print(f"ffn_output shape:         {ffn_output.shape}")
print(f"After residual addition:  {residual_2.shape}")

# LayerNorm
H_out = layer_norm_2(residual_2)  # [1, 10, 784]
print(f"After LayerNorm (H_out):  {H_out.shape}")

with torch.no_grad():
    sample = H_out[0, 0, :]
    print(f"\n  Post-LN mean: {sample.mean().item():.6f}")
    print(f"  Post-LN std:  {sample.std().item():.6f}")

print(f"\n✓ One complete Transformer block done!")
print(f"  Output: H_out ∈ R^{list(H_out.shape)}")
print(f"  In BERT-base, this block is repeated 12 times.")
print(f"  For this walkthrough, we use 1 block (the math is identical for stacked blocks).")

H_attn (FFN input) shape: torch.Size([1, 10, 784])
ffn_output shape:         torch.Size([1, 10, 784])
After residual addition:  torch.Size([1, 10, 784])
After LayerNorm (H_out):  torch.Size([1, 10, 784])

  Post-LN mean: 0.000000
  Post-LN std:  1.000638

✓ One complete Transformer block done!
  Output: H_out ∈ R^[1, 10, 784]
  In BERT-base, this block is repeated 12 times.
  For this walkthrough, we use 1 block (the math is identical for stacked blocks).


## Step 11: Contextual Representations

### What happens here

The output of the final Transformer block gives us **contextual representations** — each token's vector now encodes information about the **entire sequence** (via attention), not just the token itself.

### Key insight

Before the Transformer:
* Token "cat" at position 2 had a **static** embedding (same vector regardless of context)

After the Transformer:
* Token "cat" at position 2 now has a **contextualized** vector that encodes:
  * The fact that "cat" is the subject
  * That it's followed by a masked word
  * That "mat" appears later (potentially relevant)
  * Positional relationships with all other tokens

### Shape

$$\mathbf{H}_{\text{out}} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

Each of the 10 token positions now has a 784-dimensional vector that is **context-aware**.

### What happens next depends on the task:

* **Pre-training (MLM)**: Feed masked positions to a prediction head → predict original token
* **Pre-training (NSP)**: Feed `[CLS]` to a classifier → predict if sentences are consecutive
* **Fine-tuning**: Add a task-specific head on top of these representations

### 🔀 GPT Difference
GPT's contextual representations are **unidirectional** — each position only contains information from positions $$\leq i$$ (due to the causal mask). The representation at the **last** position is used for next-token prediction.

## Step 12: MLM Prediction Head

### What happens here

For Masked Language Model (MLM) pre-training, BERT applies a small **prediction head** on top of the contextual representations to map them back toward the vocabulary space. This head consists of:

1. A **Dense layer** (linear + activation): $$\mathbb{R}^d \rightarrow \mathbb{R}^d$$
2. **GELU** activation
3. **LayerNorm**

### Mathematical operation

$$\mathbf{H}_{\text{mlm}} = \text{LayerNorm}(\text{GELU}(\mathbf{H}_{\text{out}} \cdot W_{\text{dense}} + b_{\text{dense}}))$$

where:
* $$W_{\text{dense}} \in \mathbb{R}^{d \times d} = \mathbb{R}^{784 \times 784}$$
* $$b_{\text{dense}} \in \mathbb{R}^{784}$$

### Shape

$$\mathbf{H}_{\text{mlm}} \in \mathbb{R}^{B \times T \times d} = \mathbb{R}^{1 \times 10 \times 784}$$

### Why this extra layer?

The Transformer's contextual representations are optimized for **general-purpose** encoding. The MLM head provides a learned non-linear transform that adapts these representations specifically for the token prediction task. Without it, performance drops.

### 🔀 GPT Difference
GPT does **not** have this intermediate head. It directly projects from the contextual representation to vocabulary logits using a single linear layer (often with **weight tying** — reusing the token embedding matrix transposed as the output projection).

In [0]:
# ==============================================================
# STEP 12: MLM Prediction Head (Dense + GELU + LayerNorm)
# ==============================================================
# Dense layer: d -> d
mlm_dense = nn.Linear(d, d)       # [784, 784] + bias [784]
mlm_layer_norm = nn.LayerNorm(d, eps=1e-12)

print(f"MLM Dense layer: {list(mlm_dense.weight.shape)} + bias {list(mlm_dense.bias.shape)}")
print(f"MLM LayerNorm: gamma {list(mlm_layer_norm.weight.shape)}, beta {list(mlm_layer_norm.bias.shape)}")

# Forward: H_out -> Dense -> GELU -> LayerNorm
H_dense = mlm_dense(H_out)              # [1, 10, 784]
H_activated = F.gelu(H_dense)           # [1, 10, 784]
H_mlm = mlm_layer_norm(H_activated)     # [1, 10, 784]

print(f"\nH_out shape:       {H_out.shape}")
print(f"After Dense:       {H_dense.shape}")
print(f"After GELU:        {H_activated.shape}")
print(f"After LayerNorm:   {H_mlm.shape}")

print(f"\n✓ MLM head output: H_mlm ∈ R^{list(H_mlm.shape)}")
print(f"  (Ready for vocabulary projection)")

MLM Dense layer: [784, 784] + bias [784]
MLM LayerNorm: gamma [784], beta [784]

H_out shape:       torch.Size([1, 10, 784])
After Dense:       torch.Size([1, 10, 784])
After GELU:        torch.Size([1, 10, 784])
After LayerNorm:   torch.Size([1, 10, 784])

✓ MLM head output: H_mlm ∈ R^[1, 10, 784]
  (Ready for vocabulary projection)


## Step 13: Vocabulary Logits (Projection to Vocabulary Space)

### What happens here

The $$d$$-dimensional vectors are projected to the **full vocabulary size** $$V = 30{,}522$$ using a linear transformation. Each output value represents the model's raw "confidence" (logit) that a particular vocabulary token belongs at that position.

### Mathematical operation

$$\text{logits} = \mathbf{H}_{\text{mlm}} \cdot W_{\text{vocab}}^\top + b_{\text{vocab}}$$

where:
* $$W_{\text{vocab}} \in \mathbb{R}^{V \times d} = \mathbb{R}^{30522 \times 784}$$
* $$b_{\text{vocab}} \in \mathbb{R}^{V} = \mathbb{R}^{30522}$$

### Weight Tying (common optimization)

In both BERT and GPT, it's common to **tie** the output projection weights with the token embedding matrix:

$$W_{\text{vocab}} = \mathbf{E}_{\text{tok}} \in \mathbb{R}^{V \times d}$$

This means the same matrix is used for:
* **Embedding**: token ID → dense vector (forward lookup)
* **Prediction**: dense vector → vocabulary scores (matrix multiply)

This reduces parameters by $$V \times d \approx 24\text{M}$$ and acts as a regularizer.

### Shape

$$\text{logits} \in \mathbb{R}^{B \times T \times V} = \mathbb{R}^{1 \times 10 \times 30522}$$

Each position gets a score for **every word** in the vocabulary.

In [0]:
# ==============================================================
# STEP 13: Vocabulary Logits
# ==============================================================
# Option A: Separate output projection (no weight tying) — conceptually:
#   vocab_projection = nn.Linear(d, V)  →  W: [V, d] = [30522, 784] + bias [30522]
#   Parameters: d * V + V = 784 * 30522 + 30522 = 23,959,770
print(f"Option A (separate projection): W_vocab [{V}, {d}] + bias [{V}]")
print(f"Parameters if untied: {d * V + V:,}")

# Option B: Weight tying (reuse token embedding weights)
# logits = H_mlm @ E_tok.weight.T + bias
# We demonstrate both; using weight tying here:
with torch.no_grad():
    # Tied weights: use token_embedding_table.weight as projection
    logits_tied = F.linear(H_mlm, token_embedding_table.weight)  # [1, 10, 30522]
    # Add output bias
    output_bias = nn.Parameter(torch.zeros(V))
    logits = logits_tied + output_bias

print(f"\nH_mlm shape:    {H_mlm.shape}       [B, T, d]")
print(f"E_tok^T shape:  [{d}, {V}]      (transposed embedding matrix)")
print(f"Logits shape:   {logits.shape}  [B, T, V]")

print(f"\n--- Interpreting logits ---")
print(f"At position 3 (the [MASK] token), we have {V} scores:")
print(f"  logits[0, 3, :10] (first 10 vocab entries): {logits[0, 3, :10].numpy().round(4)}")
print(f"  logits[0, 3, :].shape = {logits[0, 3, :].shape}  (one score per vocab word)")
print(f"\n✓ Vocabulary logits: {list(logits.shape)}")
print(f"  (These are RAW scores, not probabilities yet)")

Option A (separate projection): W_vocab [30522, 784] + bias [30522]
Parameters if untied: 23,959,770

H_mlm shape:    torch.Size([1, 10, 784])       [B, T, d]
E_tok^T shape:  [784, 30522]      (transposed embedding matrix)
Logits shape:   torch.Size([1, 10, 30522])  [B, T, V]

--- Interpreting logits ---
At position 3 (the [MASK] token), we have 30522 scores:
  logits[0, 3, :10] (first 10 vocab entries): [ 14.5008   2.3926   8.423   -2.6286  45.4634   6.967  -34.9229   5.5893
   5.9835 -19.4011]
  logits[0, 3, :].shape = torch.Size([30522])  (one score per vocab word)

✓ Vocabulary logits: [1, 10, 30522]
  (These are RAW scores, not probabilities yet)


## Step 14: Softmax

### What happens here

The raw logits are converted into a **probability distribution** over the vocabulary using softmax. After this step, each position has a valid probability distribution that sums to 1.

### Mathematical operation

For each position $$t$$ and vocabulary index $$j$$:

$$P(\text{token}_j | \text{context}) = \text{softmax}(\text{logits})_j = \frac{e^{z_j}}{\sum_{k=1}^{V} e^{z_k}}$$

where $$z_j = \text{logits}[t, j]$$.

### Shape

$$\text{probs} \in \mathbb{R}^{B \times T \times V} = \mathbb{R}^{1 \times 10 \times 30522}$$

(Same shape as logits, but now values are in $$[0, 1]$$ and each row sums to 1)

### Important notes

1. **During training**: We do NOT actually compute softmax + argmax. Instead, we use **cross-entropy loss** (which combines log-softmax and NLL loss) for numerical stability
2. **During inference**: Softmax gives us the probability of each token, then we pick the top one
3. **Temperature**: During generation (GPT), we can scale logits by a temperature $$\tau$$ before softmax:
   $$P_j = \frac{e^{z_j / \tau}}{\sum_k e^{z_k / \tau}}$$
   * $$\tau < 1$$: sharper distribution (more confident)
   * $$\tau > 1$$: flatter distribution (more creative/random)
   * $$\tau = 1$$: standard softmax

In [0]:
# ==============================================================
# STEP 14: Softmax
# ==============================================================
# Apply softmax over vocabulary dimension (last dim)
probs = F.softmax(logits, dim=-1)  # [1, 10, 30522]

print(f"Logits shape: {logits.shape}")
print(f"Probs shape:  {probs.shape}")

# Verify it's a valid probability distribution
with torch.no_grad():
    row_sum = probs[0, 3, :].sum().item()
    print(f"\nAt position 3 ([MASK]):")
    print(f"  Sum of probabilities: {row_sum:.6f} (should be 1.0)")
    print(f"  Min probability:  {probs[0, 3, :].min().item():.2e}")
    print(f"  Max probability:  {probs[0, 3, :].max().item():.4f}")
    print(f"  Mean probability: {probs[0, 3, :].mean().item():.2e} (= 1/{V} = {1/V:.2e})")

    # Top 5 most probable tokens at the masked position
    top5_probs, top5_indices = torch.topk(probs[0, 3, :], k=5)
    print(f"\n  Top 5 predictions for [MASK] position:")
    print(f"  {'Rank':<5} {'Token ID':<10} {'Probability':<12}")
    print(f"  {'-'*30}")
    for rank, (prob, idx) in enumerate(zip(top5_probs, top5_indices), 1):
        print(f"  {rank:<5} {idx.item():<10} {prob.item():.6f}")
    
    print(f"\n  (With random weights, predictions are near-uniform random)")
    print(f"  (After training, the correct token would have high probability)")

print(f"\n✓ Softmax complete: probs ∈ R^{list(probs.shape)}")

Logits shape: torch.Size([1, 10, 30522])
Probs shape:  torch.Size([1, 10, 30522])

At position 3 ([MASK]):
  Sum of probabilities: 1.000000 (should be 1.0)
  Min probability:  0.00e+00
  Max probability:  0.5046
  Mean probability: 3.28e-05 (= 1/30522 = 3.28e-05)

  Top 5 predictions for [MASK] position:
  Rank  Token ID   Probability 
  ------------------------------
  1     6264       0.504579
  2     27490      0.494616
  3     22297      0.000742
  4     652        0.000051
  5     1127       0.000005

  (With random weights, predictions are near-uniform random)
  (After training, the correct token would have high probability)

✓ Softmax complete: probs ∈ R^[1, 10, 30522]


## Step 15: Predicted Masked Token

### What happens here

The final step: take the **argmax** of the probability distribution to get the predicted token ID.

### Mathematical operation

$$\hat{y}_t = \arg\max_{j \in [0, V)} P(\text{token}_j | \text{context})$$

### During pre-training (loss computation)

We don't just take the argmax — we compute the **cross-entropy loss** between the predicted distribution and the true token:

$$\mathcal{L}_{\text{MLM}} = -\frac{1}{|M|}\sum_{t \in M} \log P(y_t | \text{context})$$

where $$M$$ is the set of masked positions and $$y_t$$ is the true token at position $$t$$.

In our example, position 3 is `[MASK]` (the original token was "sat", say), so:

$$\mathcal{L} = -\log P(\text{"sat"} | \text{context at position 3})$$

### During inference

Simply take $$\arg\max$$ to get the most probable token.

### 🔀 GPT Difference
GPT predicts the **next** token (not a masked token):

$$\mathcal{L}_{\text{LM}} = -\sum_{t=1}^{T} \log P(x_t | x_1, \ldots, x_{t-1})$$

Every position predicts the **next** token in the sequence (shifted by 1). This is why GPT can generate text autoregressively: each new token is conditioned on all previous tokens.

In [0]:
# ==============================================================
# STEP 15: Predicted Masked Token
# ==============================================================
with torch.no_grad():
    # --- Inference: argmax prediction ---
    predicted_ids = torch.argmax(probs, dim=-1)  # [1, 10]
    print(f"Predicted token IDs: {predicted_ids}")
    print(f"Shape: {predicted_ids.shape}  [B, T]")
    
    # Focus on the masked position (position 3)
    masked_pos = 3
    predicted_token_id = predicted_ids[0, masked_pos].item()
    predicted_prob = probs[0, masked_pos, predicted_token_id].item()
    
    print(f"\n--- Prediction at [MASK] position ({masked_pos}) ---")
    print(f"  Predicted token ID: {predicted_token_id}")
    print(f"  Confidence: {predicted_prob:.6f}")
    print(f"  (Low confidence expected with random weights!)")
    
    # --- Training: Cross-entropy loss ---
    # Suppose the true token at position 3 was "sat" (ID = 2938)
    true_token_id = 2938
    print(f"\n--- Pre-training loss computation ---")
    print(f"  True token at position {masked_pos}: ID = {true_token_id}")
    
    # Cross-entropy loss (only at masked positions)
    loss_at_mask = -torch.log(probs[0, masked_pos, true_token_id])
    print(f"  Loss = -log(P(true_token)) = -log({probs[0, masked_pos, true_token_id].item():.6e})")
    print(f"       = {loss_at_mask.item():.4f}")
    print(f"  (Random baseline loss ≈ -log(1/V) = log({V}) = {np.log(V):.2f})")
    
    # Using PyTorch's built-in cross-entropy
    ce_loss = F.cross_entropy(
        logits[0, masked_pos:masked_pos+1, :],  # [1, V]
        torch.tensor([true_token_id])             # [1]
    )
    print(f"  PyTorch CE loss (numerically stable): {ce_loss.item():.4f}")

print(f"\n✓ End-to-end forward pass complete!")
print(f"  Input:  token_ids {list(token_ids.shape)}")
print(f"  Output: predicted_ids {list(predicted_ids.shape)}")

Predicted token IDs: tensor([[ 9504, 23658, 11935,  6264, 10390, 10632, 16690, 25261, 26009, 22195]])
Shape: torch.Size([1, 10])  [B, T]

--- Prediction at [MASK] position (3) ---
  Predicted token ID: 6264
  Confidence: 0.504579
  (Low confidence expected with random weights!)

--- Pre-training loss computation ---
  True token at position 3: ID = 2938
  Loss = -log(P(true_token)) = -log(0.000000e+00)
       = inf
  (Random baseline loss ≈ -log(1/V) = log(30522) = 10.33)
  PyTorch CE loss (numerically stable): 150.3584

✓ End-to-end forward pass complete!
  Input:  token_ids [1, 10]
  Output: predicted_ids [1, 10]


## Overall: Consolidated Modularized PyTorch Implementation

---

### Summary of the Complete Forward Pass

| Step | Operation | Input Shape | Output Shape | Key Parameters |
|------|-----------|-------------|--------------|----------------|
| 1 | Token IDs | raw text | $$[1, 10]$$ | (tokenizer) |
| 2 | Token Embedding | $$[1, 10]$$ | $$[1, 10, 784]$$ | $$30522 \times 784 \approx 24\text{M}$$ |
| 3 | Position Embedding | $$[1, 10]$$ | $$[1, 10, 784]$$ | $$10 \times 784 = 7{,}840$$ |
| 4 | Token-Type Embedding | $$[1, 10]$$ | $$[1, 10, 784]$$ | $$2 \times 784 = 1{,}568$$ |
| 5 | Sum + LayerNorm | 3 × $$[1,10,784]$$ | $$[1, 10, 784]$$ | $$2 \times 784 = 1{,}568$$ |
| 6 | Q, K, V Projection | $$[1, 10, 784]$$ | 3 × $$[1, 8, 10, 98]$$ | $$3 \times (784^2 + 784) \approx 1.8\text{M}$$ |
| 7 | Self-Attention | Q,K,V | $$[1, 10, 784]$$ | $$784^2 + 784 \approx 615\text{K}$$ |
| 8 | Residual + LayerNorm | $$[1, 10, 784]$$ | $$[1, 10, 784]$$ | $$2 \times 784 = 1{,}568$$ |
| 9 | FFN (784→3136→784) | $$[1, 10, 784]$$ | $$[1, 10, 784]$$ | $$\approx 4.9\text{M}$$ |
| 10 | Residual + LayerNorm | $$[1, 10, 784]$$ | $$[1, 10, 784]$$ | $$2 \times 784 = 1{,}568$$ |
| 11 | Contextual Reps | — | $$[1, 10, 784]$$ | (output of block) |
| 12 | MLM Head | $$[1, 10, 784]$$ | $$[1, 10, 784]$$ | $$784^2 + 784 + 2 \times 784 \approx 616\text{K}$$ |
| 13 | Vocab Projection | $$[1, 10, 784]$$ | $$[1, 10, 30522]$$ | tied with Step 2 |
| 14 | Softmax | $$[1, 10, 30522]$$ | $$[1, 10, 30522]$$ | 0 |
| 15 | argmax | $$[1, 10, 30522]$$ | $$[1, 10]$$ | 0 |

---

### BERT vs GPT — Side-by-Side

| Aspect | BERT | GPT |
|--------|------|-----|
| Direction | Bidirectional (sees full sequence) | Unidirectional (causal mask) |
| Token-Type Embedding | Yes (segment A/B) | No |
| Pre-training task | MLM + NSP | Next-token prediction |
| Attention mask | None (full attention) | Lower-triangular (causal) |
| Prediction head | Dense + GELU + LN + Projection | Direct projection (weight tied) |
| Generation | Cannot generate (fills blanks) | Autoregressive generation |
| LayerNorm placement | Post-LN | Pre-LN (GPT-2+) |

---

Below is the **complete, modularized PyTorch code** consolidating all steps into proper classes.

In [0]:
# ==============================================================
# OVERALL: Complete Modularized PyTorch Implementation
# ==============================================================
# A clean, self-contained BERT-style Transformer for MLM
# with GPT-style causal variant clearly marked.
# ==============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ========================== CONFIG ==========================
class TransformerConfig:
    """All hyperparameters in one place."""
    def __init__(
        self,
        vocab_size: int = 30522,
        max_seq_length: int = 10,
        hidden_dim: int = 784,
        num_heads: int = 8,
        ff_dim: int = 3136,
        num_layers: int = 1,       # Using 1 for demonstration
        num_segments: int = 2,
        dropout: float = 0.1,
        layer_norm_eps: float = 1e-12,
        is_causal: bool = False,   # False=BERT, True=GPT
    ):
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.ff_dim = ff_dim
        self.num_layers = num_layers
        self.num_segments = num_segments
        self.dropout = dropout
        self.layer_norm_eps = layer_norm_eps
        self.is_causal = is_causal


# ======================== EMBEDDINGS ========================
class TransformerEmbeddings(nn.Module):
    """
    Steps 1-5: Token + Position + Segment embeddings, then LayerNorm + Dropout.
    
    GPT variant: set use_segment_embedding=False.
    """
    def __init__(self, config: TransformerConfig, use_segment_embedding: bool = True):
        super().__init__()
        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_dim)
        self.position_embedding = nn.Embedding(config.max_seq_length, config.hidden_dim)
        self.use_segment_embedding = use_segment_embedding
        if use_segment_embedding:
            self.segment_embedding = nn.Embedding(config.num_segments, config.hidden_dim)
        self.layer_norm = nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, token_ids, segment_ids=None):
        B, T = token_ids.shape
        
        # Step 2: Token embedding lookup
        tok_emb = self.token_embedding(token_ids)           # [B, T, d]
        
        # Step 3: Position embedding lookup
        position_ids = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb = self.position_embedding(position_ids)     # [1, T, d]
        
        # Step 4: Segment embedding (BERT only)
        embeddings = tok_emb + pos_emb
        if self.use_segment_embedding and segment_ids is not None:
            seg_emb = self.segment_embedding(segment_ids)   # [B, T, d]
            embeddings = embeddings + seg_emb
        
        # Step 5: LayerNorm + Dropout
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings  # [B, T, d]


# =================== MULTI-HEAD ATTENTION ===================
class MultiHeadSelfAttention(nn.Module):
    """
    Steps 6-7: Q/K/V projections + Scaled Dot-Product Attention.
    
    Supports both bidirectional (BERT) and causal (GPT) attention.
    """
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.num_heads = config.num_heads
        self.head_dim = config.head_dim
        self.is_causal = config.is_causal
        
        # Step 6: Q, K, V linear projections
        self.W_Q = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.W_K = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.W_V = nn.Linear(config.hidden_dim, config.hidden_dim)
        
        # Output projection
        self.W_O = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, X, attention_mask=None):
        B, T, d = X.shape
        
        # Step 6: Linear projections
        Q = self.W_Q(X)  # [B, T, d]
        K = self.W_K(X)
        V = self.W_V(X)
        
        # Reshape to multi-head: [B, T, d] -> [B, h, T, d_k]
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Step 7a: Scaled dot-product attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # [B, h, T, d_k] x [B, h, d_k, T] = [B, h, T, T]
        
        # Step 7b: Apply causal mask (GPT) or padding mask
        if self.is_causal:
            causal_mask = torch.triu(
                torch.ones(T, T, device=X.device), diagonal=1
            ).bool()
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        if attention_mask is not None:
            # attention_mask: [B, 1, 1, T] with 0 for valid, -inf for padding
            scores = scores + attention_mask
        
        # Step 7c: Softmax
        attn_weights = F.softmax(scores, dim=-1)  # [B, h, T, T]
        attn_weights = self.dropout(attn_weights)
        
        # Step 7d: Weighted sum of values
        attn_output = torch.matmul(attn_weights, V)  # [B, h, T, d_k]
        
        # Step 7e: Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, d)
        output = self.W_O(attn_output)  # [B, T, d]
        
        return output, attn_weights


# ===================== FEED-FORWARD ======================
class FeedForwardNetwork(nn.Module):
    """
    Step 9: Two-layer MLP with GELU activation.
    784 -> 3136 -> 784
    """
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_dim, config.ff_dim)   # d -> d_ff
        self.linear_2 = nn.Linear(config.ff_dim, config.hidden_dim)   # d_ff -> d
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.linear_1(x)       # [B, T, d_ff]
        x = F.gelu(x)             # GELU activation
        x = self.linear_2(x)       # [B, T, d]
        x = self.dropout(x)
        return x


# ================== TRANSFORMER BLOCK ====================
class TransformerBlock(nn.Module):
    """
    Steps 6-10: One complete Transformer encoder/decoder block.
    Attention -> Residual+LN -> FFN -> Residual+LN
    """
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.attention = MultiHeadSelfAttention(config)
        self.ffn = FeedForwardNetwork(config)
        self.layer_norm_1 = nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
        self.layer_norm_2 = nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, X, attention_mask=None):
        # Steps 7-8: Self-attention + Residual + LayerNorm
        attn_output, attn_weights = self.attention(X, attention_mask)
        H_attn = self.layer_norm_1(X + attn_output)          # Post-LN
        
        # Steps 9-10: FFN + Residual + LayerNorm
        ffn_output = self.ffn(H_attn)
        H_out = self.layer_norm_2(H_attn + ffn_output)       # Post-LN
        
        return H_out, attn_weights  # [B, T, d], [B, h, T, T]


# ================== MLM PREDICTION HEAD ====================
class MLMHead(nn.Module):
    """
    Steps 12-15: Dense + GELU + LN + Vocab Projection + Softmax.
    Uses weight tying with the token embedding matrix.
    """
    def __init__(self, config: TransformerConfig, embedding_weights: nn.Parameter):
        super().__init__()
        self.dense = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.layer_norm = nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
        self.output_bias = nn.Parameter(torch.zeros(config.vocab_size))
        # Weight tying: reuse embedding matrix for output projection
        self.embedding_weights = embedding_weights  # [V, d]

    def forward(self, hidden_states):
        # Step 12: Dense + GELU + LayerNorm
        h = self.dense(hidden_states)       # [B, T, d]
        h = F.gelu(h)
        h = self.layer_norm(h)              # [B, T, d]
        
        # Step 13: Project to vocabulary (weight-tied)
        logits = F.linear(h, self.embedding_weights) + self.output_bias  # [B, T, V]
        
        return logits  # Raw logits (use cross-entropy loss for training)


# ================== FULL MODEL ====================
class BERTForMLM(nn.Module):
    """
    Complete BERT-style model for Masked Language Modeling.
    Consolidates all 15 steps into a single forward pass.
    """
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config
        
        # Steps 1-5: Embeddings
        self.embeddings = TransformerEmbeddings(
            config, 
            use_segment_embedding=not config.is_causal  # GPT has no segments
        )
        
        # Steps 6-10: Transformer blocks (stacked N times)
        self.encoder_blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.num_layers)
        ])
        
        # Steps 12-15: MLM prediction head (weight-tied)
        self.mlm_head = MLMHead(
            config, 
            self.embeddings.token_embedding.weight  # Weight tying!
        )

    def forward(self, token_ids, segment_ids=None, attention_mask=None, masked_positions=None):
        """
        Full forward pass from token IDs to vocabulary predictions.
        
        Args:
            token_ids:       [B, T] integer token IDs
            segment_ids:     [B, T] segment IDs (0 or 1), BERT only
            attention_mask:  [B, T] with 1 for valid tokens, 0 for padding
            masked_positions: [B, M] indices of masked positions (optional, for efficiency)
        
        Returns:
            logits: [B, T, V] or [B, M, V] vocabulary logits
        """
        # Steps 1-5: Compute input embeddings
        X = self.embeddings(token_ids, segment_ids)  # [B, T, d]
        
        # Prepare attention mask if provided
        if attention_mask is not None:
            # Convert [B, T] -> [B, 1, 1, T] for broadcasting
            extended_mask = attention_mask[:, None, None, :].float()
            extended_mask = (1.0 - extended_mask) * -1e9
        else:
            extended_mask = None
        
        # Steps 6-10: Pass through all Transformer blocks
        H = X
        all_attention_weights = []
        for block in self.encoder_blocks:
            H, attn_weights = block(H, extended_mask)
            all_attention_weights.append(attn_weights)
        
        # Step 11: H is now the contextual representation [B, T, d]
        
        # Steps 12-15: MLM head -> logits
        logits = self.mlm_head(H)  # [B, T, V]
        
        return logits, all_attention_weights

    def predict_masked_tokens(self, token_ids, segment_ids=None, masked_positions=None):
        """Convenience method: returns predicted token IDs and probabilities."""
        self.eval()
        with torch.no_grad():
            logits, _ = self.forward(token_ids, segment_ids)
            probs = F.softmax(logits, dim=-1)        # Step 14
            predicted_ids = torch.argmax(probs, dim=-1)  # Step 15
        return predicted_ids, probs


# ==============================================================
# INSTANTIATE AND RUN
# ==============================================================
print("=" * 60)
print("COMPLETE MODEL INSTANTIATION & FORWARD PASS")
print("=" * 60)

# Create config
config = TransformerConfig(
    vocab_size=30522,
    max_seq_length=10,
    hidden_dim=784,
    num_heads=8,
    ff_dim=3136,
    num_layers=1,
    is_causal=False  # BERT mode
)

# Instantiate model
model = BERTForMLM(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

# Prepare inputs
token_ids = torch.tensor([[101, 1996, 4937, 103, 2006, 1996, 13523, 102, 2009, 102]])
segment_ids = torch.tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1]])

print(f"\n{'='*60}")
print(f"FORWARD PASS")
print(f"{'='*60}")
print(f"Input token_ids:  {token_ids.shape} = {token_ids.tolist()}")
print(f"Input segment_ids: {segment_ids.shape} = {segment_ids.tolist()}")

# Forward pass
logits, attn_weights = model(token_ids, segment_ids)
print(f"\nOutput logits shape: {logits.shape}  [B, T, V]")
print(f"Attention weights:   {attn_weights[0].shape}  [B, h, T, T]")

# Predict masked tokens
predicted_ids, probs = model.predict_masked_tokens(token_ids, segment_ids)
print(f"\nPredicted IDs: {predicted_ids}")
print(f"\n--- Masked position 3 ([MASK]) ---")
top5_probs, top5_ids = torch.topk(probs[0, 3, :], k=5)
print(f"  Top 5 predictions:")
for i, (p, idx) in enumerate(zip(top5_probs, top5_ids), 1):
    print(f"    {i}. Token ID {idx.item():>6} | P = {p.item():.6f}")

# Training loss example
true_labels = torch.tensor([[101, 1996, 4937, 2938, 2006, 1996, 13523, 102, 2009, 102]])
# Only compute loss at masked positions (position 3)
mask_indices = torch.tensor([3])
loss = F.cross_entropy(
    logits[0, mask_indices, :],           # [M, V]
    true_labels[0, mask_indices]           # [M]
)
print(f"\n--- Training loss at masked position ---")
print(f"  Cross-entropy loss: {loss.item():.4f}")
print(f"  (Expected ≈ {math.log(config.vocab_size):.2f} for random model)")

print(f"\n{'='*60}")
print(f"✓ COMPLETE! Full Transformer forward pass demonstrated.")
print(f"{'='*60}")

COMPLETE MODEL INSTANTIATION & FORWARD PASS

Total parameters: 31,973,818

Model architecture:
BERTForMLM(
  (embeddings): TransformerEmbeddings(
    (token_embedding): Embedding(30522, 784)
    (position_embedding): Embedding(10, 784)
    (segment_embedding): Embedding(2, 784)
    (layer_norm): LayerNorm((784,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder_blocks): ModuleList(
    (0): TransformerBlock(
      (attention): MultiHeadSelfAttention(
        (W_Q): Linear(in_features=784, out_features=784, bias=True)
        (W_K): Linear(in_features=784, out_features=784, bias=True)
        (W_V): Linear(in_features=784, out_features=784, bias=True)
        (W_O): Linear(in_features=784, out_features=784, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForwardNetwork(
        (linear_1): Linear(in_features=784, out_features=3136, bias=True)
        (linear_2): Linear(in_features=3136, out_features=784,

## Key Takeaways

---

### The Transformer is fundamentally a sequence of matrix multiplications:

1. **Embedding layers** → Table lookups (index operations, no matmul)
2. **Q, K, V projections** → $$[B, T, d] \times [d, d] = [B, T, d]$$
3. **Attention scores** → $$[B, h, T, d_k] \times [B, h, d_k, T] = [B, h, T, T]$$
4. **Attention output** → $$[B, h, T, T] \times [B, h, T, d_k] = [B, h, T, d_k]$$
5. **FFN Layer 1** → $$[B, T, d] \times [d, d_{ff}] = [B, T, d_{ff}]$$
6. **FFN Layer 2** → $$[B, T, d_{ff}] \times [d_{ff}, d] = [B, T, d]$$
7. **Vocab projection** → $$[B, T, d] \times [d, V] = [B, T, V]$$

Everything else (softmax, GELU, LayerNorm, residuals) is **element-wise** or **reduction** operations.

---

### The Single Difference Between BERT and GPT (architecturally):

> **Attention masking.** BERT sees the full sequence bidirectionally. GPT applies a causal (lower-triangular) mask so each position can only attend to earlier positions.

All other differences (no segment embedding, no MLM head, pre-LN vs post-LN) are **design choices**, not fundamental architectural changes.

---

### Parameter count breakdown (1 layer, our config):

| Component | Formula | Count |
|-----------|---------|-------|
| Token Embedding | $$V \times d$$ | 23,929,248 |
| Position Embedding | $$T \times d$$ | 7,840 |
| Segment Embedding | $$2 \times d$$ | 1,568 |
| Q, K, V + Output projections | $$4 \times (d^2 + d)$$ | 2,462,336 |
| FFN | $$2 \times d \times d_{ff} + d_{ff} + d$$ | 4,921,168 |
| LayerNorms (×3) | $$3 \times 2d$$ | 4,704 |
| MLM Head | $$d^2 + d + 2d + V$$ (tied) | 647,432 |
| **Total** | | **∲31.9M** |